## Run Evaluation

In [1]:
import os
import time
import json
import codecs
import pickle
from typing import Any, Dict, List

import re
from bs4 import BeautifulSoup

from agents.llm_model import UnifiedModel, model_name


# Base folder for data
DATA_DIR = "data"

# Folder to store pickle outputs
OUT_ROOT = os.path.join(DATA_DIR, "judge_pickles_anthropic")

# Files to evaluate (relative to DATA_DIR)
FILES_TO_EVAL = [
    "en_longInputD2T_LLMtexts_inputs100.json",
    "en_longInputD2T_LLMtexts_inputs95.json",
    "en_longInputD2T_LLMtexts_inputs90.json",
    "en_longInputD2T_LLMtexts_inputs85.json",
    "ga_longInputD2T_LLMtexts_inputs100.json",
    "ga_longInputD2T_LLMtexts_inputs95.json",
    "ga_longInputD2T_LLMtexts_inputs90.json",
    "ga_longInputD2T_LLMtexts_inputs85.json",
]

# Language labels for the prompt
LANG_LABEL = {
    "en": "English",
    "ga": "Irish (Gaeilge)",
}

# Identifier to use in the pickle key: scores_{MODEL_TAG}
MODEL_TAG = "claude_anthropic"

# How long to sleep between API calls (seconds)
SLEEP_SECONDS = 10

In [2]:
# BASE_PROMPT = """
# In this task, you will evaluate the quality of the Text in relation to the given Triple Set. How well does the Text represent the Triple Set? You will be given five specific Dimensions to evaluate against:

# Dimensions:
# No-Omissions: To what degree is ALL the information in the Triple Set present in the Text?
# No-Additions: To what degree is ONLY information from the Triple Set present in the Text?
# Grammaticality: To what degree is the Text free of grammatical errors, looking at its form only?
# Coherence: To what degree is the Text well-structured and well-organised into a coherent body of information about a topic, from the perspective of meaning only?
# Fluency: To what degree does the Text flow, and can be absorbed readily without bringing the reader up short?

# Important note on No-Omissions and No-Additions: Whether there are omissions and or additions in a Text is NOT related to factual truth, but instead is strictly related to the contents of the input Triple Set.
# Important note on Grammaticality, Coherence and Fluency: for Grammaticality, Coherence and Fluency you do not need to consider the input Triple Set. only the intrinsic quality of the Text needs to be assessed.

# You need to provide the scores ranging from 1 (indicating the lowest score) to 5 (indicating the highest score) for each of the Dimensions and a short justification for each score in the following JSON format:
# {{
#   "No-Omissions": {{"Justification": "", "Score": ""}},
#   "No-Additions": {{"Justification": "", "Score": ""}},
#   "Grammaticality": {{"Justification": "", "Score": ""}},
#   "Coherence": {{"Justification": "", "Score": ""}},
#   "Fluency": {{"Justification": "", "Score": ""}}
# }}

# Make sure to read thoroughly the Triple Set and the {Language} Text below, and assess the five Dimensions using the instructions and template above.

# Triple Set:
# {Triples}

# Text:
# {Input_Text}
# """

In [ ]:
BASE_PROMPT = """
You are evaluating how well a Text realises a given Triple Set.

Your task:
1. Read the Triple Set and the {Language} Text carefully.
2. For each of the five Dimensions below, assign a score from 1 (lowest) to 5 (highest).
3. For each Dimension, give a short justification (one or two sentences).
4. Return only a single JSON object in the exact format specified. Do not include any extra text.

Dimensions:
No-Omissions: To what degree is ALL the information in the Triple Set present in the Text.
No-Additions: To what degree does the Text include ONLY information from the Triple Set.
Grammaticality: To what degree is the Text free of grammatical errors, considering form only.
Coherence: To what degree is the Text well structured and organised into a coherent body of information about a topic, from the perspective of meaning only.
Fluency: To what degree does the Text read smoothly and naturally, without abrupt or awkward phrasing.

Important notes:
- No-Omissions and No-Additions are judged only with respect to the Triple Set. Factual correctness beyond the triples does not matter.
- Grammaticality, Coherence and Fluency are intrinsic properties of the Text. You do not need the Triple Set to judge them.
- Scores must be integers in the set (1, 2, 3, 4, 5).

Return your assessment in this exact JSON format, with no additional keys and no extra text:
{{
  "No-Omissions": {{"Justification": "", "Score": ""}},
  "No-Additions": {{"Justification": "", "Score": ""}},
  "Grammaticality": {{"Justification": "", "Score": ""}},
  "Coherence": {{"Justification": "", "Score": ""}},
  "Fluency": {{"Justification": "", "Score": ""}}
}}

Triple Set:
{Triples}

Text:
{Input_Text}
"""

In [4]:
def language_label_from_filename(fname: str) -> str:
    """
    Infer language label from file name.
    """
    base = os.path.basename(fname)
    if base.startswith("en_"):
        return LANG_LABEL["en"]
    if base.startswith("ga_"):
        return LANG_LABEL["ga"]
    return "the given"


def build_prompt(
    prompt_draft: str,
    triples: str,
    text: str,
    language_label: str,
) -> str:
    """
    Build the LLM as judge prompt given the triple set string,
    the generated text, and a human readable language label.
    """
    prompt = prompt_draft.format(
        Triples=triples,
        Input_Text=text,
        Language=language_label,
    )
    return prompt.strip()


def make_judge_chain():
    """
    Build a LangChain runnable that calls Claude via UnifiedModel.
    """
    cfg = model_name.get("anthropic", {}).copy()
    if "temperature" not in cfg:
        cfg["temperature"] = 0.0

    system_prompt = (
        "You are an expert evaluator that strictly follows the user instructions "
        "and returns only valid JSON in the requested format."
    )

    return UnifiedModel(provider="anthropic", **cfg).model_(system_prompt)


In [5]:
def format_json(json_path: str) -> List[Dict[str, Any]]:
    """
    Convert the original JSON file into a list of dicts with keys:
      id, triples, text

    The input HTML in the 'input' field is parsed as a table.
    """
    en_regular_json = json.load(codecs.open(json_path, "r", "utf-8"))

    triples_text_pairs: List[Dict[str, Any]] = []
    x = 0
    while x < len(en_regular_json):
        html = en_regular_json[x]["input"]
        soup = BeautifulSoup(html, "html.parser")
        table = soup.find("table")

        rows: List[str] = []
        if table is not None:
            for row in table.find_all("tr"):
                columns = row.find_all(["td", "th"])
                row_data = " ".join(col.text.strip() for col in columns)
                rows.append(row_data)

        triples_formatted = "; ".join(rows[1:]) if len(rows) > 1 else ""
        triples_text_pairs.append(
            {
                "id": en_regular_json[x]["id"],
                "triples": f'"""{triples_formatted}"""',
                "text": en_regular_json[x]["output"],
            }
        )
        x += 1

    return triples_text_pairs

In [6]:
def main():
    os.makedirs(OUT_ROOT, exist_ok=True)
    chain = make_judge_chain()

    for fname in FILES_TO_EVAL:
        in_path = os.path.join(DATA_DIR, fname)
        if not os.path.exists(in_path):
            print(f"[WARN] Input file not found: {in_path}. skipping.")
            continue

        lang_label = language_label_from_filename(fname)
        base_name = os.path.splitext(os.path.basename(fname))[0]
        out_folder = os.path.join(OUT_ROOT, base_name)
        os.makedirs(out_folder, exist_ok=True)

        triples_text_pairs = format_json(in_path)
        n_examples = len(triples_text_pairs)

        # Existing pickle files in this folder
        existing_files = set(os.listdir(out_folder))
        existing_pickles = {
            f for f in existing_files if f.startswith(base_name) and f.endswith(".pkl")
        }

        # If we already have at least one pickle per example, skip the whole file
        if len(existing_pickles) >= n_examples:
            print(
                f"\n[SKIP] {in_path} already has {len(existing_pickles)} pickles "
                f"for {n_examples} examples in {out_folder}. skipping file."
            )
            continue

        print(f"\nEvaluating file: {in_path} ({n_examples} examples)")
        print(f"Results will be stored as pickles in: {out_folder}/")

        for i, dp in enumerate(triples_text_pairs):
            ex_id = dp["id"]
            triples = dp["triples"]
            text = dp["text"]

            pickle_name = f"{base_name}_{ex_id}.pkl"
            out_path = os.path.join(out_folder, pickle_name)

            # Skip this example if its pickle already exists
            if os.path.exists(out_path):
                print(
                    f"[SKIP] #{i+1}/{n_examples} for model {MODEL_TAG} "
                    f"(id={ex_id}) already has pickle {pickle_name}. skipping."
                )
                continue

            print(
                f"Evaluating #{i+1}/{n_examples} for model {MODEL_TAG} "
                f"(id={ex_id})..."
            )

            prompt = build_prompt(
                prompt_draft=BASE_PROMPT,
                triples=triples,
                text=text,
                language_label=lang_label,
            )

            try:
                resp = chain.invoke({"input": prompt})
                response_content = getattr(resp, "content", resp)
            except Exception as e:
                print(f"[ERROR] Call failed for id={ex_id}: {e}")
                response_content = f"ERROR: {e}"

            record = {
                "id": ex_id,
                "file": fname,
                "language_label": lang_label,
                "triples": triples,
                "text": text,
                f"scores_{MODEL_TAG}": response_content,
            }

            with open(out_path, "wb") as f:
                pickle.dump(record, f)

            time.sleep(SLEEP_SECONDS)

        print(f"Results for {fname} saved in {out_folder}/")

    print("\nAll judge evaluations completed.")


In [ ]:
main()

## Results analysis

In [8]:
#!/usr/bin/env python
import os
import re
import json
import ast
import pickle
from typing import Any, Dict, DefaultDict
from collections import defaultdict

# Root directory that contains your subfolders
# en_longInputD2T_LLMtexts_inputs100, ga_longInputD2T_LLMtexts_inputs95, etc.
PICKLE_ROOT = "data/judge_pickles_anthropic_2"


def parse_scores_dict(raw: Any) -> Dict[str, Any]:
    """
    Try to normalise and parse the judge output into a Python dict.
    Accepts plain strings or Anthropic style message lists.
    """
    if isinstance(raw, list):
        try:
            raw = "".join(getattr(x, "text", str(x)) for x in raw)
        except Exception:
            raw = "".join(str(x) for x in raw)

    s = str(raw)

    # Remove trivial wrappers
    s = s.replace("```json", "").replace("```", "")
    s = s.replace("None", "null")

    # Clip to outermost JSON object
    start = s.find("{")
    end = s.rfind("}")
    if start != -1 and end != -1:
        s = s[start : end + 1]

    # First try JSON
    try:
        d = json.loads(s)
    except Exception:
        # Fallback to literal eval
        try:
            d = ast.literal_eval(s)
        except Exception:
            d = {}

    if isinstance(d, dict) and "query" in d:
        d = d["query"]

    return d if isinstance(d, dict) else {}


def get_float_score(d: Dict[str, Any], dim: str) -> float:
    """
    Extract a numeric score for a given dimension key.
    Expected structure: d[dim]["Score"] in {1..5} as string or number.
    """
    try:
        val = d.get(dim, {}).get("Score", 0)
        return float(val)
    except Exception:
        return 0.0


def collect_scores_for_folder(folder_path: str) -> Dict[int, Dict[str, float]]:
    """
    Aggregate scores per model index for all pkl files in a single folder.

    Returns:
      {model_idx: {count, no_omissions, no_additions, grammaticality, coherence, fluency}}
      where values are averages.
    """
    sums: DefaultDict[int, Dict[str, float]] = defaultdict(
        lambda: {
            "no_omissions": 0.0,
            "no_additions": 0.0,
            "grammaticality": 0.0,
            "coherence": 0.0,
            "fluency": 0.0,
            "count": 0,
        }
    )

    files = sorted(f for f in os.listdir(folder_path) if f.endswith(".pkl"))
    if not files:
        print(f"[INFO] No pickle files in {folder_path}. skipping.")
        return {}

    for fname in files:
        # Last number before .pkl is the model index
        m = re.search(r"_(\d+)\.pkl$", fname)
        if not m:
            print(f"[WARN] Could not extract model index from {fname}")
            continue
        model_idx = int(m.group(1))

        path = os.path.join(folder_path, fname)
        try:
            with open(path, "rb") as f:
                rec = pickle.load(f)
        except Exception as e:
            print(f"[WARN] Could not load {path}: {e}")
            continue

        # Find the scores_* key
        score_key = next((k for k in rec.keys() if k.startswith("scores_")), None)
        if not score_key:
            print(f"[WARN] No scores_* key in {path}")
            continue

        raw_scores = rec[score_key]
        d = parse_scores_dict(raw_scores)
        if not d:
            print(f"[WARN] Could not parse scores in {path}")
            continue

        acc = sums[model_idx]
        acc["no_omissions"] += get_float_score(d, "No-Omissions")
        acc["no_additions"] += get_float_score(d, "No-Additions")
        acc["grammaticality"] += get_float_score(d, "Grammaticality")
        acc["coherence"] += get_float_score(d, "Coherence")
        acc["fluency"] += get_float_score(d, "Fluency")
        acc["count"] += 1

    # Convert sums to averages
    averages: Dict[int, Dict[str, float]] = {}
    for model_idx, vals in sums.items():
        c = vals["count"]
        if c == 0:
            continue
        averages[model_idx] = {
            "count": c,
            "no_omissions": vals["no_omissions"] / c,
            "no_additions": vals["no_additions"] / c,
            "grammaticality": vals["grammaticality"] / c,
            "coherence": vals["coherence"] / c,
            "fluency": vals["fluency"] / c,
        }

    return averages


def main():
    if not os.path.isdir(PICKLE_ROOT):
        print(f"[ERROR] Pickle root folder not found: {PICKLE_ROOT}")
        return

    # Process each immediate subfolder separately
    for folder_name in sorted(os.listdir(PICKLE_ROOT)):
        folder_path = os.path.join(PICKLE_ROOT, folder_name)
        if not os.path.isdir(folder_path):
            continue

        print(f"\nProcessing folder: {folder_path}")
        averages = collect_scores_for_folder(folder_path)

        if not averages:
            print(f"[INFO] No usable scores in {folder_path}")
            continue

        # Print summary
        for model_idx in sorted(averages.keys()):
            stats = averages[model_idx]
            print(
                f"  Model {model_idx}: "
                # f"count={stats['count']}, "
                f"Avg_No-Omissions={stats['no_omissions']:.3f}, "
                f"Avg_No-Additions={stats['no_additions']:.3f}, "
                f"Avg_Grammaticality={stats['grammaticality']:.3f}, "
                f"Avg_Fluency={stats['fluency']:.3f}, "
                f"Avg_Coherence={stats['coherence']:.3f}"
            )

        # Save per folder JSON into that folder
        out_json = os.path.join("results", f"{folder_name}_model_averages_2.json")
        with open(out_json, "w", encoding="utf-8") as f:
            json.dump(averages, f, indent=2, ensure_ascii=False)
        print(f"  Saved averages to {out_json}")


if __name__ == "__main__":
    main()



Processing folder: data/judge_pickles_anthropic_2/en_longInputD2T_LLMtexts_inputs100
  Model 0: Avg_No-Omissions=4.833, Avg_No-Additions=4.933, Avg_Grammaticality=5.000, Avg_Fluency=4.533, Avg_Coherence=4.700
  Model 1: Avg_No-Omissions=4.967, Avg_No-Additions=4.967, Avg_Grammaticality=5.000, Avg_Fluency=4.733, Avg_Coherence=4.767
  Model 2: Avg_No-Omissions=4.933, Avg_No-Additions=4.933, Avg_Grammaticality=5.000, Avg_Fluency=4.767, Avg_Coherence=4.900
  Model 3: Avg_No-Omissions=4.733, Avg_No-Additions=4.967, Avg_Grammaticality=5.000, Avg_Fluency=4.433, Avg_Coherence=4.767
  Model 4: Avg_No-Omissions=4.533, Avg_No-Additions=4.967, Avg_Grammaticality=4.933, Avg_Fluency=4.567, Avg_Coherence=4.800
  Model 5: Avg_No-Omissions=4.000, Avg_No-Additions=4.933, Avg_Grammaticality=4.900, Avg_Fluency=4.533, Avg_Coherence=4.700
  Saved averages to results/en_longInputD2T_LLMtexts_inputs100_model_averages_2.json

Processing folder: data/judge_pickles_anthropic_2/en_longInputD2T_LLMtexts_inputs85


In [ ]:
c.i

## Ranking

In [6]:
import argparse
import html
import json
import math
import os
import re
import time
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
from openai import OpenAI

try:
    from pydantic import BaseModel, Field
except Exception as e:
    raise RuntimeError("Please install pydantic: pip install pydantic") from e


class CandidateScore(BaseModel):
    system_id: int = Field(..., description="The numeric system id, one of 0, 1, 2.")
    no_omissions: int = Field(..., ge=1, le=5)
    no_additions: int = Field(..., ge=1, le=5)
    coherence: int = Field(..., ge=1, le=5)
    fluency: int = Field(..., ge=1, le=5)
    grammaticality: int = Field(..., ge=1, le=5)
    overall: float = Field(..., ge=1, le=5)
    short_notes: List[str] = Field(..., description="Up to 3 short notes, each under 20 words.")


class RankingOutput(BaseModel):
    best: int = Field(..., description="System id ranked 1st, one of 0, 1, 2.")
    second: int = Field(..., description="System id ranked 2nd, one of 0, 1, 2.")
    third: int = Field(..., description="System id ranked 3rd, one of 0, 1, 2.")
    scores: List[CandidateScore]
    brief_rationale: str = Field(..., description="At most 120 words, British English.")


@dataclass
class Record:
    rid: str
    base_id: str
    sys_id: int
    input_html: str
    output_text: str


def parse_id(rid: str) -> Tuple[str, int]:
    m = re.search(r"_(\d+)$", rid)
    if not m:
        raise ValueError(f"Could not parse system suffix from id: {rid}")
    sys_id = int(m.group(1))
    base_id = rid[: m.start() + 1]
    return base_id, sys_id


def extract_triples_from_html_table(table_html: str) -> List[Tuple[str, str, str]]:
    cells = re.findall(r"<td>(.*?)</td>", table_html, flags=re.DOTALL | re.IGNORECASE)
    cleaned = []
    for c in cells:
        t = re.sub(r"\s+", " ", c).strip()
        t = html.unescape(t)
        cleaned.append(t)
    triples = []
    for i in range(0, len(cleaned), 2):
        if i + 2 >= len(cleaned):
            break
        s, p, o = cleaned[i], cleaned[i + 1], cleaned[i + 2]
        triples.append((s, p, o))
    return triples


def triples_to_text(triples: List[Tuple[str, str, str]], max_triples: Optional[int] = None) -> str:
    use_triples = triples if max_triples is None else triples[:max_triples]
    lines = []
    for s, p, o in use_triples:
        lines.append(f"{s} | {p} | {o}")
    return "\n".join(lines)


def cosine_sim(a: np.ndarray, b: np.ndarray) -> float:
    na = np.linalg.norm(a)
    nb = np.linalg.norm(b)
    if na == 0.0 or nb == 0.0:
        return 0.0
    return float(np.dot(a, b) / (na * nb))


def load_records(path: str) -> List[Record]:
    with open(path, "r", encoding="utf-8") as f:
        raw = json.load(f)
    out = []
    for r in raw:
        rid = r["id"]
        base_id, sys_id = parse_id(rid)
        out.append(
            Record(
                rid=rid,
                base_id=base_id,
                sys_id=sys_id,
                input_html=r["input"],
                output_text=r["output"],
            )
        )
    return out


def group_by_base(records: List[Record]) -> Dict[str, Dict[int, Record]]:
    grouped: Dict[str, Dict[int, Record]] = {}
    for rec in records:
        grouped.setdefault(rec.base_id, {})
        grouped[rec.base_id][rec.sys_id] = rec
    return grouped


def batched(lst: List[Any], n: int) -> List[List[Any]]:
    return [lst[i : i + n] for i in range(0, len(lst), n)]


def load_jsonl_map(path: str) -> Dict[str, Any]:
    if not os.path.exists(path):
        return {}
    m: Dict[str, Any] = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            m[obj["key"]] = obj["value"]
    return m


def append_jsonl(path: str, rows: List[Dict[str, Any]]) -> None:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "a", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


def get_embeddings_with_cache(
    client: OpenAI,
    model: str,
    items: List[Tuple[str, str]],
    cache_path: str,
    batch_size: int = 64,
    sleep_s: float = 0.2,
) -> Dict[str, List[float]]:
    cache = load_jsonl_map(cache_path)
    missing = [(k, t) for (k, t) in items if k not in cache]

    for chunk in batched(missing, batch_size):
        keys = [k for k, _ in chunk]
        texts = [t for _, t in chunk]
        resp = client.embeddings.create(model=model, input=texts)
        rows = []
        for k, emb in zip(keys, resp.data):
            rows.append({"key": k, "value": emb.embedding})
        append_jsonl(cache_path, rows)
        for r in rows:
            cache[r["key"]] = r["value"]
        time.sleep(sleep_s)

    return cache


def rank_three_candidates(
    client: OpenAI,
    judge_model: str,
    triple_text: str,
    candidates: Dict[int, str],
    temperature: float = 0.0,
    max_output_tokens: int = 700,
) -> RankingOutput:
    # sys_prompt = (    "You are an expert evaluator for data-to-text generation.\n"
    #     "You will be given a triple set and three candidate texts.\n"
    #     "Rank the candidates 1st, 2nd, 3rd using these criteria:\n"
    #     "1) No omissions: all facts from the triple set are present.\n"
    #     "2) No additions: no extra facts beyond the triple set.\n"
    #     "3) Coherence: well structured, logical flow.\n"
    #     "4) Fluency: reads naturally.\n"
    #     "5) Grammaticality: grammar, spelling, punctuation.\n"
    #     "Use scores 1 to 5 for each criterion. Then output an overall score 1 to 5.\n"
    #     "Return strict JSON matching the requested schema.")
    
    sys_prompt = """You are an expert evaluator for data-to-text generation.

Your task is to assess how well a candidate Text realises a given Triple Set. You will score each candidate on five dimensions and then rank the three candidates as 1st, 2nd, and 3rd.

Dimensions (score each from 1 to 5):
1. No-Omissions: To what extent is all information expressed in the Triple Set present in the Text.
2. No-Additions: To what extent does the Text contain only information supported by the Triple Set, with no extra facts.
3. Grammaticality: To what extent is the Text free of grammatical errors, considering form only.
4. Coherence: To what extent is the Text well structured and well organised into a clear, coherent description of the topic, considering meaning and organisation only.
5. Fluency: To what extent does the Text read smoothly and naturally.

Critical notes:
- No-Omissions and No-Additions are judged strictly against the provided Triple Set. Do not use outside knowledge. Do not reward plausible but unsupported statements.
- Grammaticality, Coherence, and Fluency are intrinsic text quality judgements. Do not use the Triple Set when scoring these three dimensions, except to understand what the text is about.
- Be consistent across candidates. If two texts make the same error, score them similarly.
- Prefer faithfulness over style. A fluent hallucination should score poorly on No-Additions.

Ranking rule:
- Rank primarily by No-Additions and No-Omissions, then by Coherence, Fluency, and Grammaticality as tie breakers.
- Use the full 1 to 5 scale where appropriate.

Output:
- Return strict JSON matching the required schema."""


    cand_block = []
    for sid in sorted(candidates.keys()):
        cand_block.append(f"SYSTEM {sid} TEXT:\n{candidates[sid]}")
    cand_text = "\n\n".join(cand_block)

    user_prompt = (
        "TRIPLE SET (Subject | Predicate | Object):\n"
        f"{triple_text}\n\n"
        f"{cand_text}\n\n"
        "Task:\n"
        "1) Score each system on each criterion (1-5).\n"
        "2) Provide best, second, third.\n"
        "Constraints:\n"
        "- best, second, third must be a permutation of (0,1,2).\n"
        "- brief_rationale must be at most 120 words.\n"
        "- short_notes: up to 3 notes per candidate, each under 20 words.\n"
    )

    if hasattr(client.responses, "parse"):
        resp = client.responses.parse(
            model=judge_model,
            input=[
                {"role": "system", "content": sys_prompt},
                {"role": "user", "content": user_prompt},
            ],
            text_format=RankingOutput,
            temperature=temperature,
            max_output_tokens=max_output_tokens,
        )
        return resp.output_parsed

    schema = RankingOutput.model_json_schema() if hasattr(RankingOutput, "model_json_schema") else RankingOutput.schema()
    resp = client.responses.create(
        model=judge_model,
        input=[
            {"role": "system", "content": sys_prompt},
            {"role": "user", "content": user_prompt},
        ],
        text={
            "format": {
                "type": "json_schema",
                "name": "d2t_ranking",
                "strict": True,
                "schema": schema,
            }
        },
        temperature=temperature,
        max_output_tokens=max_output_tokens,
    )
    parsed = json.loads(resp.output_text)
    return RankingOutput(**parsed)


def summarise_rank_counts(rank_rows: List[Dict[str, Any]], systems: List[int]) -> Dict[str, Any]:
    first = {s: 0 for s in systems}
    second = {s: 0 for s in systems}
    third = {s: 0 for s in systems}

    for r in rank_rows:
        first[r["best"]] += 1
        second[r["second"]] += 1
        third[r["third"]] += 1

    return {
        "n_ranked_items": len(rank_rows),
        "first_place_counts": first,
        "second_place_counts": second,
        "third_place_counts": third,
    }


def run_file(
    input_path: str,
    out_dir: str,
    systems: List[int],
    embed_model: str,
    judge_model: str,
    max_triples_in_prompt: Optional[int],
    limit: Optional[int],
) -> None:
    client = OpenAI()

    records = load_records(input_path)
    grouped = group_by_base(records)

    os.makedirs(out_dir, exist_ok=True)
    embeddings_cache_path = os.path.join(out_dir, "embeddings_cache.jsonl")
    rankings_path = os.path.join(out_dir, "rankings.jsonl")
    summary_path = os.path.join(out_dir, "summary.json")

    done = load_jsonl_map(rankings_path)
    rank_rows: List[Dict[str, Any]] = []

    base_ids = sorted(grouped.keys())
    if limit is not None:
        base_ids = base_ids[:limit]

    embed_items: List[Tuple[str, str]] = []
    for bid in base_ids:
        recs = grouped[bid]
        if not all(s in recs for s in systems):
            continue
        for s in systems:
            k = f"{bid}__sys{s}"
            embed_items.append((k, recs[s].output_text))
        triples = extract_triples_from_html_table(recs[systems[0]].input_html)
        triple_text = triples_to_text(triples, max_triples=max_triples_in_prompt)
        embed_items.append((f"{bid}__input", triple_text))

    emb_map = get_embeddings_with_cache(
        client=client,
        model=embed_model,
        items=embed_items,
        cache_path=embeddings_cache_path,
        batch_size=64,
        sleep_s=0.2,
    )

    new_rank_rows_to_append: List[Dict[str, Any]] = []

    for bid in base_ids:
        if bid in done:
            rank_rows.append(done[bid])
            continue

        recs = grouped[bid]
        if not all(s in recs for s in systems):
            continue

        triples = extract_triples_from_html_table(recs[systems[0]].input_html)
        triple_text = triples_to_text(triples, max_triples=max_triples_in_prompt)

        candidates = {s: recs[s].output_text for s in systems}

        input_vec = np.array(emb_map[f"{bid}__input"], dtype=np.float32)
        sims = {}
        for s in systems:
            out_vec = np.array(emb_map[f"{bid}__sys{s}"], dtype=np.float32)
            sims[str(s)] = cosine_sim(input_vec, out_vec)

        ranking = rank_three_candidates(
            client=client,
            judge_model=judge_model,
            triple_text=triple_text,
            candidates=candidates,
            temperature=0.0,
            max_output_tokens=700,
        )

        row = {
            "base_id": bid,
            "best": ranking.best,
            "second": ranking.second,
            "third": ranking.third,
            "scores": [sc.model_dump() if hasattr(sc, "model_dump") else sc.dict() for sc in ranking.scores],
            "brief_rationale": ranking.brief_rationale,
            "input_output_cosine_sims": sims,
        }

        new_rank_rows_to_append.append({"key": bid, "value": row})
        rank_rows.append(row)

        if len(new_rank_rows_to_append) >= 10:
            append_jsonl(rankings_path, new_rank_rows_to_append)
            new_rank_rows_to_append = []

    if new_rank_rows_to_append:
        append_jsonl(rankings_path, new_rank_rows_to_append)

    summary = summarise_rank_counts(rank_rows, systems)
    with open(summary_path, "w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)

    print("Done.")
    print(f"Rankings: {rankings_path}")
    print(f"Summary:  {summary_path}")
    print(f"Embeddings cache: {embeddings_cache_path}")




run_file(
    input_path="data/en_longInputD2T_LLMtexts_inputs100.json",
    out_dir="data/results_rank_en",
    systems=[0, 1, 2],
    embed_model="text-embedding-3-large",
    judge_model="gpt-5.1",
    max_triples_in_prompt=None,
    limit=None
)
run_file(
    input_path="data/ga_longInputD2T_LLMtexts_inputs100.json",
    out_dir="data/results_rank_ga",
    systems=[0, 1, 2],
    embed_model="text-embedding-3-large",
    judge_model="gpt-5.1",
    max_triples_in_prompt=None,
    limit=None
)


Done.
Rankings: data/results_rank_en/rankings.jsonl
Summary:  data/results_rank_en/summary.json
Embeddings cache: data/results_rank_en/embeddings_cache.jsonl
Done.
Rankings: data/results_rank_ga/rankings.jsonl
Summary:  data/results_rank_ga/summary.json
Embeddings cache: data/results_rank_ga/embeddings_cache.jsonl


In [7]:
import json
import os
from collections import defaultdict

import pandas as pd


def load_rankings_jsonl(path: str):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            rows.append(obj["value"])
    return rows


def summarise_from_rankings(rank_rows, systems=(0, 1, 2)):
    systems = list(systems)

    place_counts = {
        "first": {s: 0 for s in systems},
        "second": {s: 0 for s in systems},
        "third": {s: 0 for s in systems},
    }

    score_sums = {s: defaultdict(float) for s in systems}
    score_counts = {s: defaultdict(int) for s in systems}

    sim_sums = {s: 0.0 for s in systems}
    sim_counts = {s: 0 for s in systems}

    for r in rank_rows:
        place_counts["first"][r["best"]] += 1
        place_counts["second"][r["second"]] += 1
        place_counts["third"][r["third"]] += 1

        scores = r.get("scores", [])
        for sc in scores:
            sid = sc["system_id"]
            if sid not in systems:
                continue
            for k, v in sc.items():
                if k in ("system_id", "short_notes"):
                    continue
                score_sums[sid][k] += float(v)
                score_counts[sid][k] += 1

        sims = r.get("input_output_cosine_sims", {})
        for sid in systems:
            v = sims.get(str(sid), None)
            if v is None:
                continue
            sim_sums[sid] += float(v)
            sim_counts[sid] += 1

    summary_rows = []
    for sid in systems:
        row = {
            "system": sid,
            "n_items": len(rank_rows),
            "first": place_counts["first"][sid],
            "second": place_counts["second"][sid],
            "third": place_counts["third"][sid],
            "win_rate": place_counts["first"][sid] / len(rank_rows) if rank_rows else 0.0,
            "avg_input_cosine": (sim_sums[sid] / sim_counts[sid]) if sim_counts[sid] else None,
        }

        for metric in ["no_omissions", "no_additions", "coherence", "fluency", "grammaticality", "overall"]:
            c = score_counts[sid].get(metric, 0)
            row[f"avg_{metric}"] = (score_sums[sid][metric] / c) if c else None

        summary_rows.append(row)

    df_summary = pd.DataFrame(summary_rows).sort_values(["first", "avg_overall"], ascending=False)

    df_places = pd.DataFrame(
        [
            {"place": "first", **place_counts["first"]},
            {"place": "second", **place_counts["second"]},
            {"place": "third", **place_counts["third"]},
        ]
    )

    return df_summary, df_places


def run_results(lang: str, out_dir: str, systems=(0, 1, 2)):
    rankings_path = os.path.join(out_dir, "rankings.jsonl")
    summary_path = os.path.join(out_dir, "summary.json")

    print(f"\nLanguage: {lang}")
    print(f"Reading: {rankings_path}")

    rank_rows = load_rankings_jsonl(rankings_path)
    print(f"Ranked samples: {len(rank_rows)}")

    if os.path.exists(summary_path):
        with open(summary_path, "r", encoding="utf-8") as f:
            tool_summary = json.load(f)
        print("Existing summary.json (from your script):")
        print(tool_summary)

    df_summary, df_places = summarise_from_rankings(rank_rows, systems=systems)

    print("\nPlace counts:")
    display(df_places)

    print("\nPer-system results:")
    display(df_summary)

    out_csv = os.path.join(out_dir, "results_summary.csv")
    df_summary.to_csv(out_csv, index=False)
    print(f"\nSaved: {out_csv}")

    return df_summary, df_places, rank_rows


df_en, places_en, rows_en = run_results("en", "data/results_rank_en", systems=(0, 1, 2))
df_ga, places_ga, rows_ga = run_results("ga", "data/results_rank_ga", systems=(0, 1, 2))


Language: en
Reading: data/results_rank_en/rankings.jsonl
Ranked samples: 30
Existing summary.json (from your script):
{'n_ranked_items': 30, 'first_place_counts': {'0': 3, '1': 15, '2': 12}, 'second_place_counts': {'0': 2, '1': 11, '2': 17}, 'third_place_counts': {'0': 25, '1': 4, '2': 1}}

Place counts:


,place,0,1,2
0,first,3,15,12
1,second,2,11,17
2,third,25,4,1



Per-system results:


,system,n_items,first,second,third,win_rate,avg_input_cosine,avg_no_omissions,avg_no_additions,avg_coherence,avg_fluency,avg_grammaticality,avg_overall
1,1,30,15,11,4,0.5,0.744783,4.533333,3.933333,4.900000,4.900000,4.900000,4.633333
2,2,30,12,17,1,0.4,0.743518,4.600000,3.700000,4.966667,4.966667,4.966667,4.636667
0,0,30,3,2,25,0.1,0.750626,4.400000,3.266667,4.733333,4.733333,4.866667,4.380000



Saved: data/results_rank_en/results_summary.csv

Language: ga
Reading: data/results_rank_ga/rankings.jsonl
Ranked samples: 30
Existing summary.json (from your script):
{'n_ranked_items': 30, 'first_place_counts': {'0': 2, '1': 9, '2': 19}, 'second_place_counts': {'0': 5, '1': 15, '2': 10}, 'third_place_counts': {'0': 23, '1': 6, '2': 1}}

Place counts:


,place,0,1,2
0,first,2,9,19
1,second,5,15,10
2,third,23,6,1



Per-system results:


,system,n_items,first,second,third,win_rate,avg_input_cosine,avg_no_omissions,avg_no_additions,avg_coherence,avg_fluency,avg_grammaticality,avg_overall
2,2,30,19,10,1,0.633333,0.612000,4.533333,3.533333,4.800000,4.800000,4.933333,4.510000
1,1,30,9,15,6,0.300000,0.612418,4.433333,3.666667,4.800000,4.766667,4.900000,4.500000
0,0,30,2,5,23,0.066667,0.597899,4.366667,3.200000,4.566667,4.600000,4.933333,4.306667



Saved: data/results_rank_ga/results_summary.csv


In [8]:
import pandas as pd

def flatten_rank_rows(lang, rank_rows):
    flat = []
    for r in rank_rows:
        base_id = r["base_id"]
        flat.append(
            {
                "lang": lang,
                "base_id": base_id,
                "best": r["best"],
                "second": r["second"],
                "third": r["third"],
                "cos_0": r.get("input_output_cosine_sims", {}).get("0", None),
                "cos_1": r.get("input_output_cosine_sims", {}).get("1", None),
                "cos_3": r.get("input_output_cosine_sims", {}).get("3", None),
            }
        )
    return flat

combined = pd.DataFrame(flatten_rank_rows("en", rows_en) + flatten_rank_rows("ga", rows_ga))
combined_path = "data/combined_rankings_overview.csv"
combined.to_csv(combined_path, index=False)
print(f"Saved: {combined_path}")
display(combined.head(10))


Saved: data/combined_rankings_overview.csv


,lang,base_id,best,second,third,cos_0,cos_1,cos_3
0,en,en_D2T-1-FA_0000_,2,1,0,0.671365,0.678527,None
1,en,en_D2T-1-FA_0013_,2,1,0,0.744627,0.769423,None
2,en,en_D2T-1-FA_0025_,1,2,0,0.709366,0.719566,None
3,en,en_D2T-1-FA_0034_,1,2,0,0.773574,0.753352,None
4,en,en_D2T-1-FA_0052_,1,2,0,0.695793,0.736734,None
5,en,en_D2T-1-FA_0053_,1,2,0,0.757205,0.779856,None
6,en,en_D2T-1-FA_0064_,1,2,0,0.743318,0.741475,None
7,en,en_D2T-1-FA_0068_,1,2,0,0.654766,0.715241,None
8,en,en_D2T-1-FA_0074_,1,2,0,0.713468,0.740465,None
9,en,en_D2T-1-FA_0076_,1,2,0,0.815514,0.750658,None


In [9]:
import json
from collections import Counter

def load_rankings_jsonl(path: str):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            rows.append(obj["value"])
    return rows

def placement_percentages(rank_rows, systems=(0, 1, 2)):
    systems = list(systems)
    n = len(rank_rows)
    counts_best = Counter()
    counts_second = Counter()
    counts_third = Counter()

    for r in rank_rows:
        counts_best[r["best"]] += 1
        counts_second[r["second"]] += 1
        counts_third[r["third"]] += 1

    def to_percent(counter):
        out = []
        for s in systems:
            c = int(counter.get(s, 0))
            pct = (c / n * 100.0) if n else 0.0
            out.append({"system": s, "count": c, "percent": pct})
        return out

    return {
        "n_ranked_items": n,
        "best": to_percent(counts_best),
        "second": to_percent(counts_second),
        "third": to_percent(counts_third),
    }

def print_placement_percentages(path: str, systems=(0, 1, 2), label: str = ""):
    rows = load_rankings_jsonl(path)
    stats = placement_percentages(rows, systems=systems)

    if label:
        print(f"\n{label}")
    print(f"Ranked items: {stats['n_ranked_items']}")

    for place in ["best", "second", "third"]:
        print(f"\n{place.upper()}")
        for r in stats[place]:
            s = r["system"]
            c = r["count"]
            p = r["percent"]
            print(f"System {s}: {c} ({p:.2f}%)")

    return stats

stats_en = print_placement_percentages(
    "data/results_rank_en/rankings.jsonl",
    systems=(0, 1, 2),
    label="English"
)

stats_ga = print_placement_percentages(
    "data/results_rank_ga/rankings.jsonl",
    systems=(0, 1, 2),
    label="Irish"
)



English
Ranked items: 30

BEST
System 0: 3 (10.00%)
System 1: 15 (50.00%)
System 2: 12 (40.00%)

SECOND
System 0: 2 (6.67%)
System 1: 11 (36.67%)
System 2: 17 (56.67%)

THIRD
System 0: 25 (83.33%)
System 1: 4 (13.33%)
System 2: 1 (3.33%)

Irish
Ranked items: 30

BEST
System 0: 2 (6.67%)
System 1: 9 (30.00%)
System 2: 19 (63.33%)

SECOND
System 0: 5 (16.67%)
System 1: 15 (50.00%)
System 2: 10 (33.33%)

THIRD
System 0: 23 (76.67%)
System 1: 6 (20.00%)
System 2: 1 (3.33%)
